In [ ]:
import os
import warnings
import logging
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
# %pip install xgboost sktime deep-forest scikit-optimize

# Suppress verbose logging
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")
logging.getLogger("tensorflow").setLevel(logging.ERROR)

import sys
try:
    dataset_name = os.listdir('/kaggle/input/datasets/keithmarange')[0]
    sys.path.append(f'/kaggle/input/datasets/keithmarange/{dataset_name}/')
    sys.path.append('/kaggle/input/cmi-competition-code')
except Exception as e:
    pass

from sklearn.model_selection import GroupKFold, GridSearchCV, GroupShuffleSplit
from sklearn.metrics import f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier

try:
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    SKOPT_AVAILABLE = False

# Attempt to import deep-forest (install via: pip install deep-forest)
try:
    from deepforest import CascadeForestClassifier
    DEEP_FOREST_AVAILABLE = True
except ImportError:
    DEEP_FOREST_AVAILABLE = False
    print("Warning: deep-forest not installed. Install via 'pip install deep-forest'")

# Custom utilities
import data_utils
from v1_vibration_extractor import V1CargoExtractor

from base_utils_qwen import competition_scorer as bfrb_competition_scorer, evaluate_holdout, plot_training_curves
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, make_scorer

import xgboost as xgb

# Add this to your existing imports in Cell 1
try:
    from sktime.transformations.panel.rocket import MiniRocket
    from sklearn.linear_model import RidgeClassifier
    ROCKET_AVAILABLE = True
except ImportError:
    ROCKET_AVAILABLE = False
    print("Warning: sktime not installed. Install via 'pip install sktime'")

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from baselines_utils import (
    ManyToOneWrapper, 
    ManyToOneWrapperTemporal, 
    make_baseline_pipeline, 
    build_feature_extractor
)

In [ ]:
# ============================================================
# CONFIGURATION — switch feature + classifier pipelines here
# ============================================================
TRAIN_DUMMY = False
TRAIN_RF = True
TRAIN_XGB = True
TRAIN_DEEP_FOREST = True
# Add this alongside TRAIN_RF, TRAIN_XGB, etc.
TRAIN_ROCKET = True

# Feature extraction mode
FEATURE_MODE = "cargo_spectral" 

# Tabular augmentation (optional)
use_tabular_augment = False

target_col = "orientation"  # or 'bfrb' depending on your target
search_mode = "bayes"    # "grid" or "bayesian"
random_state = 42
n_splits = 2
n_iter = 15
train_size = 0.5       # lower for quick local tests; raise for full runs
error_score_constant = 0.0
verbose = 2
do_cross_val = False

if do_cross_val:
    cv_object = GroupKFold(n_splits=n_splits)
else:
    cv_object = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=random_state)

if target_col == 'bfrb':
    competition_scorer = bfrb_competition_scorer
else:
    competition_scorer = make_scorer(f1_score, average="macro", zero_division=0)

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

filter_non_brb_classes = True  
filter_orientation_class_list = None

In [ ]:
# Install all required packages
!pip install -q xgboost scikit-optimize sktime deep-forest

# Verify installations
import xgboost
import skopt
print(f"XGBoost version: {xgboost.__version__}")
print(f"scikit-optimize version: {skopt.__version__}")

In [ ]:
# ============================================================
# DATA LOADING & PREPROCESSING
# ============================================================
data_root = data_utils.find_data_root()

raw_train_df = pd.read_csv(data_root / "train.csv")
train_demo_df = pd.read_csv(data_root / "train_demographics.csv")

train_df = raw_train_df.set_index("row_id").copy(deep=True)

# Handedness correction
train_df["handedness"] = train_df["subject"].map(train_demo_df.set_index("subject")["handedness"])
left_handed_mask = train_df["handedness"].eq(0)
train_df.loc[left_handed_mask, "acc_x"] *= -1.0

# Upside-down correction
upside_down_mask = train_df["subject"].isin(["SUBJ_019262", "SUBJ_045235"])
train_df.loc[upside_down_mask, ["acc_x", "acc_y", "acc_z"]] *= -1.0
train_df.loc[upside_down_mask, ["rot_x", "rot_y", "rot_z"]] *= -1.0
train_df = train_df.drop(columns=["handedness"])

# Targets
train_df["is_target"] = (train_df["sequence_type"] == "Target").astype(bool)
train_df["bfrb"] = train_df["gesture"].where(train_df["is_target"], "non_bfrb")

train_df["gesture_position"] = train_df["gesture"].str.split(" - ").str[0]
train_df["gesture_action"] = train_df["gesture"].str.split(" - ").str[-1]


if filter_non_brb_classes:
    train_df = train_df.loc[train_df['is_target'],:]

if filter_orientation_class_list is not None:
    train_df = train_df[train_df['orientation'].isin(filter_orientation_class_list)]

    # Get unique sequences
    sequences = train_df[['sequence_id', 'is_target', 'bfrb', 'orientation', 'gesture_action']].drop_duplicates()

    # Stratified split by your target column
    train_seqs, test_seqs = train_test_split(
        sequences['sequence_id'], 
        test_size=(1 - train_size), 
        stratify=sequences[target_col],  # or use multiple columns
        random_state=random_state
    )

    train_sample_df = train_df[train_df['sequence_id'].isin(train_seqs)]
    hold_out_df = train_df[train_df['sequence_id'].isin(test_seqs)]
else:
    train_sample_df, hold_out_df = data_utils.sample_balanced_split(
        train_df, train_pct=train_size, test_pct=min(0.18, 1 - train_size), random_state=random_state
    )


X_train = train_sample_df.copy()
X_test = hold_out_df.copy()
y_train = train_sample_df[["sequence_id", "is_target", target_col]].copy()
y_test = hold_out_df[["sequence_id", "is_target", target_col]].copy()
groups = X_train["sequence_id"]

In [ ]:
# ============================================================================
# PARAMETER SPACE DEFINITION — INCLUDES FEATURE EXTRACTION & CLASSIFIERS
# ============================================================================

if search_mode == "bayesian" and SKOPT_AVAILABLE:
    
    # ===== SHARED TABULAR EXTRACTOR (Random Forest, XGBoost, Deep Forest) =====
    tabular_extractor_bayes_space = {
        "extractor__sampling_rate": Integer(20, 200),
        "extractor__acc_modes": Categorical(["raw", "raw|velocity", "smoothed|velocity|jerk", 'jerk']),
        "extractor__rotation_modes": Categorical(["quaternion", "quaternion|angular_velocity", "quaternion|euler"]),
        "extractor__tof_modes": Categorical(["sensor_stats", "pooled_stats", "sensor_stats|pooled_diff"]),
        "extractor__thm_modes": Categorical(["centered_diff", "diff", "centered", "raw"]),
        "extractor__window_size": Integer(10, 200),
        "extractor__clip_value": Categorical([None, 50.0, 100.0, 150.0]),
        "extractor__interp_mode": Categorical(["linear", "ffill"]),
        "extractor__motion_filter_mode": Categorical([None, "kalman"]),
        "extractor__use_dead_reckoning": Categorical([True, False]),
    }

    # ===== SHARED TEMPORAL EXTRACTOR (MiniRocket) =====
    temporal_extractor_bayes_space = {
        "extractor__sampling_rate": Integer(20, 200),
        "extractor__acc_modes": Categorical(["raw", "raw|velocity", "smoothed|velocity|jerk", 'jerk']),
        "extractor__rotation_modes": Categorical(["quaternion", "quaternion|angular_velocity"]),
        "extractor__tof_modes": Categorical(["pooled_stats", "sensor_stats"]),
        "extractor__thm_modes": Categorical(["centered_diff", "raw"]),
        "extractor__maxlen": Integer(30, 300),
        "extractor__window_size": Integer(10, 50),
        "extractor__clip_value": Categorical([None, 50.0, 100.0, 150.0]),
        "extractor__interp_mode": Categorical(["linear", "ffill"]),
        "extractor__motion_filter_mode": Categorical([None, "kalman"]),
        "extractor__use_dead_reckoning": Categorical([True, False]),
        "extractor__dead_reckoning_detrend": Categorical([True, False]),
        "extractor__kalman_process_noise": Real(1e-4, 1e-2, prior="log-uniform"),
        "extractor__kalman_measurement_noise": Real(1e-2, 5e-1, prior="log-uniform"),
    }

    # ===== 1. Random Forest =====
    rf_param_space = {
        **tabular_extractor_bayes_space,
        "classifier__base_estimator__n_estimators": Integer(10, 5000),
        "classifier__base_estimator__max_depth": Integer(5, 200),
        "classifier__base_estimator__min_samples_split": Integer(2, 200),
        "classifier__base_estimator__min_samples_leaf": Integer(1, 200),
        "classifier__base_estimator__max_features": Categorical(["sqrt", "log2", None]),
        "classifier__base_estimator__class_weight": Categorical(["balanced", None]),
    }

    # ===== 2. XGBoost =====
    xgb_param_space = {
        **tabular_extractor_bayes_space,
        "classifier__base_estimator__n_estimators": Integer(5, 2000),
        "classifier__base_estimator__learning_rate": Real(1e-3, 9e-1, prior="log-uniform"),
        "classifier__base_estimator__max_depth": Integer(3, 1000),
        "classifier__base_estimator__subsample": Real(0.6, 1.0, prior="uniform"),
        "classifier__base_estimator__colsample_bytree": Real(0.6, 1.0, prior="uniform"),
        "classifier__base_estimator__min_child_weight": Integer(1, 50),
    }

    # ===== 3. Deep Forest =====
    df_param_space = {
        **tabular_extractor_bayes_space,
        "classifier__base_estimator__n_estimators": Integer(2, 5000),       # Number of cascades
        "classifier__base_estimator__max_depth": Integer(5, 5000),          # Max depth of trees
        "classifier__base_estimator__n_trees": Integer(50, 5000),          # Number of trees per layer
        "classifier__base_estimator__criterion": Categorical(["gini", "entropy"]),
    }

    # ===== 4. MiniRocket =====
    rocket_param_space = {
        **temporal_extractor_bayes_space,
        "classifier__base_estimator__num_kernels": Integer(84, 15000),
        "classifier__base_estimator__alpha": Real(1e-4, 1e3, prior="log-uniform"),
        "classifier__base_estimator__feature_selection_percentile": Categorical([None, 25, 50, 75, 90]),
        "classifier__base_estimator__class_weight": Categorical(["balanced", None]),
    }

else:  # GRID SEARCH
    
    # ===== SHARED TABULAR EXTRACTOR (Random Forest, XGBoost, Deep Forest) =====
    tabular_extractor_grid_space = {
        "extractor__sampling_rate": [50],
        "extractor__acc_modes": ["smoothed|velocity|jerk"],
        "extractor__rotation_modes": ["quaternion|angular_velocity"],
        "extractor__tof_modes": ["pooled_stats"],
        "extractor__thm_modes": ["centered_diff"],
        "extractor__window_size": [20],
        "extractor__clip_value": [50.0],
        "extractor__interp_mode": ["linear"],
        "extractor__motion_filter_mode": ["kalman"],
        "extractor__use_dead_reckoning": [False],
    }

    # ===== SHARED TEMPORAL EXTRACTOR (MiniRocket) =====
    temporal_extractor_grid_space = {
        "extractor__sampling_rate": [50],
        "extractor__acc_modes": ["smoothed|velocity|jerk"],
        "extractor__rotation_modes": ["quaternion|angular_velocity"],
        "extractor__tof_modes": ["pooled_stats"],
        "extractor__thm_modes": ["centered_diff"],
        "extractor__maxlen": [150],
        "extractor__window_size": [40],
        "extractor__clip_value": [50.0],
        "extractor__interp_mode": ["linear"],
        "extractor__motion_filter_mode": ["kalman"],
        "extractor__use_dead_reckoning": [False],
        "extractor__dead_reckoning_detrend": [False],
        "extractor__kalman_process_noise": [1e-3],
        "extractor__kalman_measurement_noise": [1e-1],
    }

    # ===== 1. Random Forest =====
    rf_param_space = {
        **tabular_extractor_grid_space,
        "classifier__base_estimator__n_estimators": [10],
        "classifier__base_estimator__max_depth": [4],
        "classifier__base_estimator__min_samples_split": [3],
        "classifier__base_estimator__min_samples_leaf": [5],
        "classifier__base_estimator__max_features": ["sqrt"],
        "classifier__base_estimator__class_weight": ["balanced"],
    }

    # ===== 2. XGBoost =====
    xgb_param_space = {
        **tabular_extractor_grid_space,
        "classifier__base_estimator__n_estimators": [20],
        "classifier__base_estimator__learning_rate": [0.05],
        "classifier__base_estimator__max_depth": [4],
        "classifier__base_estimator__subsample": [0.8],
        "classifier__base_estimator__colsample_bytree": [0.8],
        "classifier__base_estimator__min_child_weight": [1],
    }

    # ===== 3. Deep Forest =====
    df_param_space = {
        **tabular_extractor_grid_space,
        "classifier__base_estimator__n_estimators": [3],
        "classifier__base_estimator__max_depth": [10],
        "classifier__base_estimator__n_trees": [10],
        "classifier__base_estimator__criterion": ["gini"],
    }

    # ===== 4. MiniRocket =====
    rocket_param_space = {
        **temporal_extractor_grid_space,
        "classifier__base_estimator__num_kernels": [200],
        "classifier__base_estimator__alpha": [1e3],
        "classifier__base_estimator__feature_selection_percentile": [50],
        "classifier__base_estimator__class_weight": ["balanced"],
    }

In [ ]:
# ============================================================
# MODEL TRAINING & EVALUATION LOOP (FIXED FOR XGBOOST/DEEP FOREST)
# ============================================================

from sklearn.preprocessing import LabelEncoder

results_list = []
fitted_models = {}

# -------------------------------------------------------------
# Helper: encode/decode for classifiers that don't support strings
# -------------------------------------------------------------
# (We'll apply this only to XGBoost and Deep Forest)

# -------------------------------------------------------------
# 1. Random Forest (works with strings)
# -------------------------------------------------------------
if TRAIN_RF:
    print("\n--- Training: Random Forest ---")
    rf_pipe = Pipeline([
        ("extractor", build_feature_extractor("tabular_honeycomb")),
        ("classifier", ManyToOneWrapper(
            base_estimator=RandomForestClassifier(random_state=random_state, n_jobs=-1),
            target=target_col
        ))
    ])

    if search_mode == "bayesian" and SKOPT_AVAILABLE:
        rf_search = BayesSearchCV(
            rf_pipe, rf_param_space, n_iter=n_iter, scoring=competition_scorer,
            cv=cv_object, n_jobs=-1, random_state=random_state,
            error_score=error_score_constant, verbose=verbose, return_train_score=True
        )
    else:
        rf_search = GridSearchCV(
            rf_pipe, rf_param_space, scoring=competition_scorer,
            cv=cv_object, n_jobs=-1, error_score=error_score_constant, verbose=verbose, return_train_score=True
        )

    rf_search.fit(X_train, y_train, groups=groups)
    fitted_models["Random Forest"] = rf_search.best_estimator_
    
    y_pred_rf = rf_search.predict(X_test)
    from base_utils_qwen import evaluate_holdout
    eval_dict_rf = evaluate_holdout(y_test, y_pred_rf, target_col=target_col, verbose=True)
    score_rf = eval_dict_rf['competition_score']
    
    print(f"Random Forest Best CV Score: {rf_search.best_score_:.4f} | Holdout Score: {score_rf:.4f}")
    print(f"Best Params: {rf_search.best_params_}")
    
    results_list.append({
        "Model": "Random Forest",
        "CV Score": rf_search.best_score_,
        "Holdout Score": score_rf,
        "Best Params": rf_search.best_params_,
    })

# -------------------------------------------------------------
# 2. XGBoost (needs label encoding)
# -------------------------------------------------------------
if TRAIN_XGB:
    print("\n--- Training: XGBoost ---")
    
    # Encode the target labels
    le = LabelEncoder()
    y_train_encoded = y_train.copy()
    y_train_encoded['bfrb'] = le.fit_transform(y_train['bfrb'])
    
    # Build pipeline with encoded labels
    xgb_pipe = Pipeline([
        ("extractor", build_feature_extractor("tabular_honeycomb")),
        ("classifier", ManyToOneWrapper(
            base_estimator=xgb.XGBClassifier(
                objective="multi:softprob", 
                eval_metric="mlogloss", 
                random_state=random_state,
                n_jobs=-1,
                use_label_encoder=False
            ),
            target=target_col
        ))
    ])

    if search_mode == "bayesian" and SKOPT_AVAILABLE:
        xgb_search = BayesSearchCV(
            xgb_pipe, xgb_param_space, n_iter=n_iter, scoring=competition_scorer,
            cv=cv_object, n_jobs=-1, random_state=random_state,
            error_score=error_score_constant, verbose=verbose, return_train_score=True
        )
    else:
        xgb_search = GridSearchCV(
            xgb_pipe, xgb_param_space, scoring=competition_scorer,
            cv=cv_object, n_jobs=-1, error_score=error_score_constant, verbose=verbose, return_train_score=True
        )

    # Fit with encoded labels (still passing groups for stratification)
    xgb_search.fit(X_train, y_train_encoded, groups=groups)
    fitted_models["XGBoost"] = xgb_search.best_estimator_
    
    # Predict and decode back to strings
    y_pred_encoded = xgb_search.predict(X_test)
    y_pred_xgb = le.inverse_transform(y_pred_encoded)
    
    eval_dict_xgb = evaluate_holdout(y_test, y_pred_xgb, target_col=target_col, verbose=True)
    score_xgb = eval_dict_xgb['competition_score']
    
    print(f"XGBoost Best CV Score: {xgb_search.best_score_:.4f} | Holdout Score: {score_xgb:.4f}")
    print(f"Best Params: {xgb_search.best_params_}")
    
    results_list.append({
        "Model": "XGBoost",
        "CV Score": xgb_search.best_score_,
        "Holdout Score": score_xgb,
        "Best Params": xgb_search.best_params_,
    })

# -------------------------------------------------------------
# 3. Deep Forest (needs label encoding)
# -------------------------------------------------------------
if TRAIN_DEEP_FOREST and DEEP_FOREST_AVAILABLE:
    print("\n--- Training: Deep Forest ---")
    
    # Encode the target labels (reuse same encoder)
    le_df = LabelEncoder()
    y_train_encoded_df = y_train.copy()
    y_train_encoded_df['bfrb'] = le_df.fit_transform(y_train['bfrb'])
    
    df_pipe = Pipeline([
        ("extractor", build_feature_extractor("tabular_honeycomb")),
        ("classifier", ManyToOneWrapper(
            base_estimator=CascadeForestClassifier(random_state=random_state, n_jobs=-1),
            target=target_col
        ))
    ])

    if search_mode == "bayesian" and SKOPT_AVAILABLE:
        df_search = BayesSearchCV(
            df_pipe, df_param_space, n_iter=n_iter, scoring=competition_scorer,
            cv=cv_object, n_jobs=-1, random_state=random_state,
            error_score=error_score_constant, verbose=verbose, return_train_score=True
        )
    else:
        df_search = GridSearchCV(
            df_pipe, df_param_space, scoring=competition_scorer,
            cv=cv_object, n_jobs=-1, error_score=error_score_constant, verbose=verbose, return_train_score=True
        )

    df_search.fit(X_train, y_train_encoded_df, groups=groups)
    fitted_models["Deep Forest"] = df_search.best_estimator_
    
    y_pred_encoded_df = df_search.predict(X_test)
    y_pred_df = le_df.inverse_transform(y_pred_encoded_df)
    
    eval_dict_df = evaluate_holdout(y_test, y_pred_df, target_col=target_col, verbose=True)
    score_df = eval_dict_df['competition_score']
    
    print(f"Deep Forest Best CV Score: {df_search.best_score_:.4f} | Holdout Score: {score_df:.4f}")
    print(f"Best Params: {df_search.best_params_}")
    
    results_list.append({
        "Model": "Deep Forest",
        "CV Score": df_search.best_score_,
        "Holdout Score": score_df,
        "Best Params": df_search.best_params_,
    })
elif TRAIN_DEEP_FOREST and not DEEP_FOREST_AVAILABLE:
    print("\nSkipping Deep Forest: 'deep-forest' package not installed.")

# -------------------------------------------------------------
# 4. MiniRocket (works with strings via RidgeClassifier)
# -------------------------------------------------------------
if TRAIN_ROCKET and ROCKET_AVAILABLE:
    print("\n--- Training: MiniRocket ---")
    rocket_pipe = make_baseline_pipeline(
        feature_mode="temporal_honeycomb",
        classifier_name="ridge_rocket",
        target_col=target_col,
        feature_kwargs={},
        classifier_kwargs={},
        random_state=random_state,
    )

    if search_mode == "bayesian" and SKOPT_AVAILABLE:
        rocket_search = BayesSearchCV(
            rocket_pipe, rocket_param_space, n_iter=n_iter, scoring=competition_scorer,
            cv=cv_object, n_jobs=-1, random_state=random_state,
            error_score=error_score_constant, verbose=verbose, return_train_score=True
        )
    else:
        rocket_search = GridSearchCV(
            rocket_pipe, rocket_param_space, scoring=competition_scorer,
            cv=cv_object, n_jobs=-1, error_score=error_score_constant, verbose=verbose, return_train_score=True
        )

    rocket_search.fit(X_train, y_train, groups=groups)
    fitted_models["MiniRocket"] = rocket_search.best_estimator_
    
    y_pred_rocket = rocket_search.predict(X_test)
    eval_dict_rocket = evaluate_holdout(y_test, y_pred_rocket, target_col=target_col, verbose=True)
    score_rocket = eval_dict_rocket['competition_score']
    
    print(f"MiniRocket Best CV Score: {rocket_search.best_score_:.4f} | Holdout Score: {score_rocket:.4f}")
    print(f"Best Params: {rocket_search.best_params_}")
    
    results_list.append({
        "Model": "MiniRocket",
        "CV Score": rocket_search.best_score_,
        "Holdout Score": score_rocket,
        "Best Params": rocket_search.best_params_,
    })
elif TRAIN_ROCKET and not ROCKET_AVAILABLE:
    print("\nSkipping MiniRocket: 'sktime' package not installed.")

In [ ]:
# ============================================================
# FINAL SUMMARY + BEST MODEL HOLDOUT EVAL
# ============================================================
results_df = pd.DataFrame(results_list).sort_values("Holdout Score", ascending=False, na_position="last")
results_df.to_csv(results_dir / f"baselines_summary_{timestamp}.csv", index=False)

print("\n" + "=" * 60)
print("BASELINES SUMMARY")
print("=" * 60)
print(results_df.to_string(index=False))

# Full report for best holdout model
if len(results_df) > 0:
    best_name = results_df.iloc[0]["Model"]
    print(f"\nDetailed holdout eval for best model: {best_name}")
    best_model = fitted_models[best_name]
    
    y_pred = best_model.predict(X_test)
    print("\nClassification Report:")
    print(classification_report(y_test[target_col], y_pred))
    
    # Optional: Feature Importance (if extractor exposes it or via permutation)
    if hasattr(best_model.named_steps['classifier'], 'feature_importances_'):
        print("\nTop 10 Feature Importances:")
        # Note: V1CargoExtractor outputs a DataFrame, we can map importances back if needed
        # This is a simplified view; full mapping requires get_feature_names_out()